# Consensus Refinement — обучение модели для свёртки геометрического консенсуса

Задача: на вход даны **разреженный** raw-warp (геометрический консенсус всех ракурсов),
карта **coverage** (где warp валиден), **depth** в target-системе, **mean(t0,t1)**,
и **static_far** (предзаполненные небо/дальние), плюс опциональная **artifact_mask**.

Выход — RGB target-кадр. Лосс — MSE (под PSNR-метрику).

Ключевое отличие от предыдущего пайплайна (RIFE refinement):
- **Нет RIFE на входе** — чтобы не наследовать его ошибки.
- **PartialConv** в энкодере — корректно сворачивает разреженный warp
  (модель не должна путать "0=чёрный" с "0=нет данных").
- **Базовая инициализация** для residual: где coverage высокий — warp, где
  только static_far валиден — static_far, иначе модель восполняет с нуля.
- Две архитектуры на сравнение: `ConsensusUNet` (Partial U-Net) и `ConsensusMARNet`
  (SPADE-модуляция признаков по coverage/depth/artifact).

## 1. Imports

In [ ]:
import os
import sys
import json
import math
import copy
import random
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image

import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


## 2. CONFIG

In [ ]:
CV_ROOT = Path(r"C:/Users/adel/Downloads/cv_dataset")

# Windows + Jupyter: DataLoader workers часто падают (cv2/np.load в subprocess)
_DEFAULT_NUM_WORKERS = 0 if sys.platform == "win32" else 4

@dataclass
class Config:
    # === Данные (готовые артефакты на диске) ===
    dataset_root: str = str(CV_ROOT / "final_dataset_v5_participants/train")
    consensus_root: str = str(CV_ROOT / "multiview_warps/train")
    baked_root: str = str(CV_ROOT / "rife_refinement_baked/train")
    rife_root: str = str(CV_ROOT / "rife_predictions_v5/train")
    static_far_root: str = str(CV_ROOT / "precomputed_static_far")
    ego_masks_root: str = str(
        Path(r"C:/Users/adel/Documents/GitHub/YA-/methods_gallery/_ego_manual_masks/masks_approved")
    )  # 766/1278, опционально

    consensus_file: str = "consensus_raw.npy"   # готовый из multiview_warps (1278 шт.)
    use_static_far: bool = False
    use_artifact_mask: bool = True
    use_anchor_frames: bool = True
    allowed_cameras: tuple = ("front", "rear")              # mean(t0,t1) на вход
    max_samples: int = 0                      # тест на 100; 0 = все ~1278

    image_h: int = 544
    image_w: int = 1024

    # === Разделение train/val ===
    val_fraction: float = 0.16

    # === Тренировка ===
    patch_size: int = 256
    batch_size: int = 16
    num_workers: int = _DEFAULT_NUM_WORKERS

    base_channels: int = 32
    out_channels: int = 3

    # Каналы входа (use_anchor_frames=True):
    #   warp_rgb           3
    #   coverage           1
    #   depth              1
    #   static_far_rgb     3
    #   static_far_mask    1
    #   artifact_mask      1
    #   mean_t0t1_rgb      3   — (t0+t1)/2, сами t0/t1 не подаём
    in_channels: int = 13

    lr: float = 2e-4
    weight_decay: float = 5e-4
    num_epochs: int = 200
    warmup_epochs: int = 5
    grad_clip: float = 1.0

    # Mixed precision (для 4060 Ti — bf16 безопаснее fp16)
    use_amp: bool = True
    amp_dtype: str = "bf16"

    # === EMA ===
    use_ema: bool = True
    ema_decay: float = 0.9995

    # === Лосс ===
    # Вес пикселей с НИЗКИМ effective_mask. =1.0 — обычный MSE.
    # Поставь >1.0 чтобы модель сильнее училась восполнять дырки.
    loss_weight_hole: float = 1.0
    loss_weight_valid: float = 1.0

    # === Логирование ===
    log_every: int = 50
    val_every: int = 1
    save_dir: str = "./checkpoints_consensus"

    seed: int = 42


CFG = Config()
Path(CFG.save_dir).mkdir(parents=True, exist_ok=True)
print("Config OK")
print(f"  dataset:   {CFG.dataset_root}")
print(f"  consensus: {CFG.consensus_root} / {CFG.consensus_file}")
print(f"  depth:     {CFG.baked_root}/<id>/d1.npy")
print(f"  ego_masks: {CFG.ego_masks_root}")
print(f"  cameras:   {CFG.allowed_cameras}")
print(f"  static_far: {'ON' if CFG.use_static_far else 'OFF'}  artifact: {'ON' if CFG.use_artifact_mask else 'OFF'}")
print(f"  max_samples: {CFG.max_samples or 'all'}")
print(f"  lidar:     splat={CFG.lidar_splat_radius} sigma={CFG.lidar_zone_sigma} min={CFG.lidar_zone_min}")
print(f"  num_workers: {CFG.num_workers}  (0 на Windows/Jupyter — стабильная загрузка)")


## 3. Dataset — загрузка из готовых папок

Сейчас читаем напрямую (без единого `.npz` bake):

| поле модели       | откуда на диске |
|-------------------|-----------------|
| `warp_rgb`        | `multiview_warps/train/<id>/consensus_raw.npy` (CHW float 0..1) |
| `coverage`        | `multiview_warps/train/<id>/coverage.npy` |
| `depth`           | `rife_refinement_baked/train/<id>/d1.npy` (target camera, метры) |
| `target_rgb`      | `final_dataset_v5_participants/train/<id>/target/<cam>.jpg` |
| `mean_t0t1_rgb`   | `(input/t0 + input/t1) / 2` — только среднее |
| `static_far_*`    | `precomputed_static_far/<id>/static_far.npz` — **выключено** |
| `artifact_mask`   | нули — **выключено** |

Индекс сэмплов — папки в `rife_refinement_baked/train/` с валидным `meta.json`.

Depth нормализуется per-sample в `[0..1]` по `depth / (depth.max() + eps)`.

In [ ]:
from lidar_density_mask import build_lidar_density


def load_lidar_trust_maps(cfg, sample_dir: Path, camera: str, target_hw) -> dict:
    return build_lidar_density(
        sample_dir, camera, timestep=cfg.lidar_timestep,
        splat_radius=cfg.lidar_splat_radius, zone_sigma=cfg.lidar_zone_sigma,
        zone_min=cfg.lidar_zone_min, min_hits_pixel=cfg.lidar_min_hits_pixel,
        target_hw=target_hw,
    )

EPS = 1e-6

try:
    import cv2
except ImportError:
    cv2 = None


def _norm_depth(d: np.ndarray) -> np.ndarray:
    d = d.astype(np.float32)
    valid = np.isfinite(d) & (d > 0)
    if valid.sum() == 0:
        return np.zeros_like(d, dtype=np.float32)
    dmax = float(d[valid].max())
    out = np.zeros_like(d, dtype=np.float32)
    out[valid] = d[valid] / (dmax + EPS)
    return out


def _chw_npy_to_hwc_u8(path: Path) -> np.ndarray:
    arr = np.load(path).astype(np.float32)
    if arr.ndim == 3 and arr.shape[0] == 3:
        arr = np.clip(arr.transpose(1, 2, 0), 0.0, 1.0)
    return (arr * 255.0).astype(np.uint8)


def _resize_hw(img: np.ndarray, target_hw, interp) -> np.ndarray:
    H, W = target_hw
    if img.shape[:2] == (H, W):
        return img
    if cv2 is None:
        raise ImportError("opencv-python нужен для resize карт")
    return cv2.resize(img, (W, H), interpolation=interp)


def _resize_maps(warp_rgb, coverage, depth, target_hw):
    warp_rgb = _resize_hw(warp_rgb, target_hw, cv2.INTER_LINEAR)
    coverage = _resize_hw(coverage, target_hw, cv2.INTER_LINEAR)
    depth = _resize_hw(depth, target_hw, cv2.INTER_NEAREST)
    return warp_rgb, coverage, depth


class ConsensusDataset(Dataset):
    def __init__(self, sample_dirs, cfg, patch_size=256, augment=True):
        self.dirs = [Path(p) for p in sample_dirs]
        self.cfg = cfg
        self.patch_size = patch_size
        self.augment = augment

    def __len__(self):
        return len(self.dirs)

    def _load(self, baked_dir: Path) -> dict:
        meta = json.loads((baked_dir / "meta.json").read_text(encoding="utf-8"))
        sid = meta["sample_id"]
        cam = meta["camera"]
        src = Path(meta.get("source_dir", Path(self.cfg.dataset_root) / sid))
        warp_dir = Path(self.cfg.consensus_root) / sid

        warp_rgb = _chw_npy_to_hwc_u8(warp_dir / self.cfg.consensus_file)
        coverage = np.load(warp_dir / "coverage.npy").astype(np.float32)
        depth = np.load(baked_dir / "d1.npy").astype(np.float32)
        depth = np.where(np.isfinite(depth), depth, 0.0).astype(np.float32)
        target_rgb = np.array(Image.open(src / "target" / f"{cam}.jpg").convert("RGB"))

        # Единый размер для батчинга (540..554 x 1024 в датасете → cfg.image_h x image_w)
        target_hw = (self.cfg.image_h, self.cfg.image_w)
        warp_rgb, coverage, depth = _resize_maps(warp_rgb, coverage, depth, target_hw)
        target_rgb = _resize_hw(target_rgb, target_hw, cv2.INTER_LINEAR)

        t0_rgb = np.array(Image.open(src / "input" / "t0" / f"{cam}.jpg").convert("RGB"))
        t1_rgb = np.array(Image.open(src / "input" / "t1" / f"{cam}.jpg").convert("RGB"))
        t0_rgb = _resize_hw(t0_rgb, target_hw, cv2.INTER_LINEAR)
        t1_rgb = _resize_hw(t1_rgb, target_hw, cv2.INTER_LINEAR)
        mean_t0t1_rgb = ((t0_rgb.astype(np.float32) + t1_rgb.astype(np.float32)) * 0.5).astype(np.uint8)

        H, W = target_hw
        sf_rgb = np.zeros((H, W, 3), dtype=np.uint8)
        sf_mask = np.zeros((H, W), dtype=np.float32)
        if self.cfg.use_static_far:
            sf_path = Path(self.cfg.static_far_root) / sid / "static_far.npz"
            if sf_path.is_file():
                z = np.load(sf_path)
                sf_mask = z["static"].astype(np.float32)
                sf_rgb = z["mean_pred"].astype(np.uint8)
                if sf_rgb.shape[:2] != (H, W):
                    sf_rgb, sf_mask, _ = _resize_maps(sf_rgb, sf_mask, sf_mask, (H, W))

        art_mask = np.zeros((H, W), dtype=np.float32)

        return {
            "warp_rgb": warp_rgb,
            "coverage": coverage,
            "depth": depth,
            "static_far_rgb": sf_rgb,
            "static_far_mask": sf_mask,
            "artifact_mask": art_mask,
            "lidar_trust": lidar_trust,
            "mean_t0t1_rgb": mean_t0t1_rgb,
            "target_rgb": target_rgb,
        }

    def __getitem__(self, idx):
        s = self._load(self.dirs[idx])

        warp = s["warp_rgb"].astype(np.float32) / 255.0
        cov = np.clip(s["coverage"], 0.0, 1.0)
        depth_n = _norm_depth(s["depth"])
        sf_rgb = s["static_far_rgb"].astype(np.float32) / 255.0
        sf_mask = np.clip(s["static_far_mask"], 0.0, 1.0)
        art_mask = np.clip(s["artifact_mask"], 0.0, 1.0)
        mean_t0t1 = s["mean_t0t1_rgb"].astype(np.float32) / 255.0
        target = s["target_rgb"].astype(np.float32) / 255.0

        H, W = warp.shape[:2]

        if self.patch_size and (H > self.patch_size or W > self.patch_size):
            ph = pw = self.patch_size
            y0 = random.randint(0, H - ph)
            x0 = random.randint(0, W - pw)
            sl = (slice(y0, y0 + ph), slice(x0, x0 + pw))
            warp = warp[sl]; cov = cov[sl]; depth_n = depth_n[sl]
            sf_rgb = sf_rgb[sl]; sf_mask = sf_mask[sl]; art_mask = art_mask[sl]
            mean_t0t1 = mean_t0t1[sl]
            target = target[sl]

        if self.augment:
            if random.random() < 0.5:
                warp = warp[:, ::-1].copy(); cov = cov[:, ::-1].copy()
                depth_n = depth_n[:, ::-1].copy()
                sf_rgb = sf_rgb[:, ::-1].copy(); sf_mask = sf_mask[:, ::-1].copy()
                art_mask = art_mask[:, ::-1].copy(); lidar_trust = lidar_trust[:, ::-1].copy()
                mean_t0t1 = mean_t0t1[:, ::-1].copy()
                target = target[:, ::-1].copy()
            if random.random() < 0.5:
                warp = warp[::-1].copy(); cov = cov[::-1].copy()
                depth_n = depth_n[::-1].copy()
                sf_rgb = sf_rgb[::-1].copy(); sf_mask = sf_mask[::-1].copy()
                art_mask = art_mask[::-1].copy(); lidar_trust = lidar_trust[::-1].copy()
                mean_t0t1 = mean_t0t1[::-1].copy()
                target = target[::-1].copy()
            k = random.randint(0, 3)
            if k:
                rot = lambda a: np.rot90(a, k).copy()
                warp = rot(warp); cov = rot(cov); depth_n = rot(depth_n)
                sf_rgb = rot(sf_rgb); sf_mask = rot(sf_mask); art_mask = rot(art_mask); lidar_trust = rot(lidar_trust)
                mean_t0t1 = rot(mean_t0t1)
                target = rot(target)

        warp_w = cov[..., None]
        sf_w = sf_mask[..., None] * (1.0 - warp_w)
        base_init = warp * warp_w + sf_rgb * sf_w
        base_mask = np.clip(cov + sf_mask * (1.0 - cov), 0.0, 1.0)

        base_init = base_init * (1.0 - art_mask[..., None])
        base_mask = base_mask * (1.0 - art_mask)

        # в дырках warp — fallback на mean(t0,t1) для residual base
        if self.cfg.use_anchor_frames:
            hole = (1.0 - base_mask)[..., None]
            base_init = base_init + mean_t0t1 * hole

        # mean(t0,t1) плотный → partial conv видит его и в дырках
        eff_mask = np.ones_like(base_mask) if self.cfg.use_anchor_frames else base_mask

        to_chw_3 = lambda a: torch.from_numpy(a.transpose(2, 0, 1)).contiguous().float()
        to_chw_1 = lambda a: torch.from_numpy(a)[None].contiguous().float()

        parts = [
            to_chw_3(warp),
            to_chw_1(cov),
            to_chw_1(depth_n),
            to_chw_3(sf_rgb),
            to_chw_1(sf_mask),
            to_chw_1(art_mask),
        ]
        if self.cfg.use_anchor_frames:
            parts.append(to_chw_3(mean_t0t1))
        inputs = torch.cat(parts, dim=0)

        return {
            "inputs": inputs,
            "effective_mask": to_chw_1(eff_mask),
            "base_init": to_chw_3(base_init),
            "base_mask": to_chw_1(base_mask),
            "target": to_chw_3(target),
        }


def discover_samples(cfg) -> list[Path]:
    baked_root = Path(cfg.baked_root)
    consensus_root = Path(cfg.consensus_root)
    dataset_root = Path(cfg.dataset_root)
    out: list[Path] = []
    for baked_dir in sorted(baked_root.iterdir()):
        if not baked_dir.is_dir() or not (baked_dir / "meta.json").is_file():
            continue
        meta = json.loads((baked_dir / "meta.json").read_text(encoding="utf-8"))
        sid = meta["sample_id"]
        cam = meta["camera"]
        src = Path(meta.get("source_dir", dataset_root / sid))
        warp_dir = consensus_root / sid
        ok = (
            (warp_dir / cfg.consensus_file).is_file()
            and (warp_dir / "coverage.npy").is_file()
            and (baked_dir / "d1.npy").is_file()
            and (src / "target" / f"{cam}.jpg").is_file()
            and (src / "input" / "t0" / f"{cam}.jpg").is_file()
            and (src / "input" / "t1" / f"{cam}.jpg").is_file()
        )
        if ok:
            out.append(baked_dir)
    if cfg.max_samples > 0 and len(out) > cfg.max_samples:
        rng = np.random.RandomState(cfg.seed)
        idx = np.arange(len(out))
        rng.shuffle(idx)
        out = [out[i] for i in idx[: cfg.max_samples]]
    return out


ALL_SAMPLES = discover_samples(CFG)
print(f"Готово к обучению: {len(ALL_SAMPLES)} сэмплов")
if ALL_SAMPLES:
    ds = ConsensusDataset(ALL_SAMPLES[:4], CFG, patch_size=CFG.patch_size, augment=True)
    b = ds[0]
    for k, v in b.items():
        print(f"  {k}: {tuple(v.shape)} {v.dtype} [{v.min():.3f}..{v.max():.3f}]")


## 2.5 LiDAR: плотность точек → зоны (не depth-маска)

Проецируем `input/lidar.npz` в **камеру сэмпла** (`meta.camera`, timestep=`target`).
Сырые hits → Gaussian blur → `lidar_trust` [0..1]. Это маска доверия для blend,
а `d1.npy` остаётся только геометрией на вход U-Net.

Параметры — в `Config` (`lidar_zone_sigma`, `lidar_splat_radius`, …).

In [ ]:
from lidar_density_mask import build_lidar_density, plot_lidar_density_debug


def _pick_probe_baked(cfg, camera: str | None = None) -> Path:
    root = Path(cfg.baked_root)
    for baked in sorted(root.iterdir()):
        if not (baked / "meta.json").is_file():
            continue
        meta = json.loads((baked / "meta.json").read_text(encoding="utf-8"))
        cam = meta.get("camera")
        if cfg.allowed_cameras and cam not in cfg.allowed_cameras:
            continue
        if camera and cam != camera:
            continue
        src = Path(meta.get("source_dir", Path(cfg.dataset_root) / meta["sample_id"]))
        if (src / "input" / "lidar.npz").is_file():
            return baked
    raise FileNotFoundError("Нет baked-сэмпла с lidar.npz для allowed_cameras")


def load_lidar_trust_maps(cfg, sample_dir: Path, camera: str, target_hw) -> dict:
    return build_lidar_density(
        sample_dir,
        camera,
        timestep=cfg.lidar_timestep,
        splat_radius=cfg.lidar_splat_radius,
        zone_sigma=cfg.lidar_zone_sigma,
        zone_min=cfg.lidar_zone_min,
        min_hits_pixel=cfg.lidar_min_hits_pixel,
        target_hw=target_hw,
    )


PROBE_BAKED = _pick_probe_baked(CFG)
_pm = json.loads((PROBE_BAKED / "meta.json").read_text(encoding="utf-8"))
_src = Path(_pm.get("source_dir", Path(CFG.dataset_root) / _pm["sample_id"]))
_cam = _pm["camera"]
_hw = (CFG.image_h, CFG.image_w)
_lm = load_lidar_trust_maps(CFG, _src, _cam, _hw)
_tgt = np.array(Image.open(_src / "target" / f"{_cam}.jpg").convert("RGB")).astype(np.float32) / 255.0
if _tgt.shape[:2] != _hw:
    _tgt = _resize_hw(_tgt, _hw, cv2.INTER_LINEAR)
_art = None
if CFG.use_artifact_mask:
    _art = _load_ego_mask(CFG, _pm["sample_id"], _cam, _hw)
_depth_b = np.load(PROBE_BAKED / "d1.npy").astype(np.float32)
_depth_b = _resize_hw(np.where(np.isfinite(_depth_b), _depth_b, 0), _hw, cv2.INTER_NEAREST)

plot_lidar_density_debug(_lm, rgb=_tgt, art=_art, depth_baked=_depth_b,
    title=f"probe {_pm['sample_id']}  cam={_cam}")
plt.show()
print(f"projected points: {_lm['n_projected']}  |  sigma={CFG.lidar_zone_sigma}  splat={CFG.lidar_splat_radius}")


## 4. PartialConv2d — корректная свёртка разреженного входа

Идея (Liu et al. 2018, *Image Inpainting for Irregular Holes*):
- Свёртка работает только над **валидными** пикселями (`x * mask`).
- Результат **нормализуется** на отношение `slide_winsize / valid_in_window`,
  чтобы значения не "разбавлялись" нулями.
- Маска обновляется: если в окне был хоть один валидный пиксель → выход валиден.

После 2-3 PartialConv-блоков с downsampling маска обычно полностью заполняется,
и дальше можно работать обычными свёртками. Поэтому PartialConv ставим только в
**энкодере**, декодер — стандартный U-Net.

In [ ]:
class PartialConv2d(nn.Conv2d):
    # Partial Convolution с явным трекингом маски валидности.
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0,
                 dilation=1, bias=True):
        super().__init__(in_ch, out_ch, kernel_size, stride=stride,
                         padding=padding, dilation=dilation, bias=bias)
        ker = self.kernel_size
        mask_ker = torch.ones(1, 1, ker[0], ker[1])
        self.register_buffer("mask_kernel", mask_ker)
        # размер окна (для нормализации)
        self.slide_winsize = float(ker[0] * ker[1])

    def forward(self, x, mask):
        # mask: [B, 1, H, W]
        with torch.no_grad():
            updated_mask = F.conv2d(
                mask, self.mask_kernel, bias=None,
                stride=self.stride, padding=self.padding, dilation=self.dilation,
            )
            mask_ratio = self.slide_winsize / (updated_mask + 1e-8)
            updated_mask = torch.clamp(updated_mask, 0.0, 1.0)
            mask_ratio = mask_ratio * updated_mask

        x_masked = x * mask  # broadcast: mask=[B,1,H,W] на x=[B,C,H,W]
        out = F.conv2d(x_masked, self.weight, bias=None,
                       stride=self.stride, padding=self.padding,
                       dilation=self.dilation)
        if self.bias is not None:
            bias = self.bias.view(1, -1, 1, 1)
            out = (out - bias) * mask_ratio + bias
            out = out * updated_mask
        else:
            out = out * mask_ratio
        return out, updated_mask


class PartialDoubleConv(nn.Module):
    # Два PartialConv 3x3 подряд + GELU (аналог ConvBlock).
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pc1 = PartialConv2d(in_ch, out_ch, 3, padding=1)
        self.pc2 = PartialConv2d(out_ch, out_ch, 3, padding=1)
        self.act = nn.GELU()

    def forward(self, x, mask):
        x, mask = self.pc1(x, mask)
        x = self.act(x)
        x, mask = self.pc2(x, mask)
        x = self.act(x)
        return x, mask


def _pconv_sanity():
    pc = PartialConv2d(3, 16, 3, padding=1)
    x = torch.randn(2, 3, 32, 32)
    # маска с большой "дырой" посередине
    m = torch.ones(2, 1, 32, 32)
    m[:, :, 10:22, 10:22] = 0.0
    y, m2 = pc(x, m)
    leak = (m2[:, :, 10:22, 10:22] > 0).float().mean().item()
    print(f"PartialConv OK: out={tuple(y.shape)}, mask leak in hole={leak:.2%}")

_pconv_sanity()

## 5. ConsensusUNet — Partial-encoder + стандартный декодер

Архитектура:
1. На вход — `[B, 13, H, W]` (warp + meta + mean(t0,t1)) + `effective_mask` `[B, 1, H, W]`.
2. **Encoder** (4 уровня) — `PartialDoubleConv`, маска проходит вниз и заполняется.
3. **Bottleneck** — обычный `ConvBlock` (маска к этому моменту почти везде = 1).
4. **Decoder** — обычные `ConvBlock` со skip-connections; на каждом уровне
   передаём также **upsampled mask** конкатом, чтобы декодер знал, где источник
   был сильнее.
5. **Head** — residual (zero-init) поверх `base_init`. На старте обучения
   `pred ≈ base_init`, идентичный warp/static_far бейзлайн.

Дополнительно — `confidence_head` (sigmoid), который можно использовать для
weighted-loss или визуализации, но в loss не идёт по умолчанию.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GELU(),
        )
    def forward(self, x):
        return self.block(x)


class ConsensusUNet(nn.Module):
    def __init__(self, in_ch=13, base=32, predict_confidence=True):
        super().__init__()
        self.predict_confidence = predict_confidence

        # Partial Encoder (4 уровня)
        self.penc1 = PartialDoubleConv(in_ch, base)
        self.penc2 = PartialDoubleConv(base, base * 2)
        self.penc3 = PartialDoubleConv(base * 2, base * 4)
        self.penc4 = PartialDoubleConv(base * 4, base * 8)

        # Bottleneck — обычная свёртка (маска уже плотная)
        self.bottleneck = ConvBlock(base * 8, base * 8)

        # Decoder: +1 канал на каждом уровне — это upsampled mask с этого уровня
        self.dec4 = ConvBlock(base * 8 + base * 8 + 1, base * 4)
        self.dec3 = ConvBlock(base * 4 + base * 4 + 1, base * 2)
        self.dec2 = ConvBlock(base * 2 + base * 2 + 1, base)
        self.dec1 = ConvBlock(base     + base     + 1, base)

        # Residual head (zero-init → identity start)
        self.out_residual = nn.Conv2d(base, 3, 3, padding=1)
        nn.init.zeros_(self.out_residual.weight)
        nn.init.zeros_(self.out_residual.bias)

        if predict_confidence:
            self.out_conf = nn.Conv2d(base, 1, 3, padding=1)
            nn.init.zeros_(self.out_conf.weight)
            nn.init.zeros_(self.out_conf.bias)

        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

    def _pool_with_mask(self, x, m):
        # признаки усредняем, маску — max (есть хоть один валидный)
        x_d = F.avg_pool2d(x, 2)
        m_d = F.max_pool2d(m, 2)
        return x_d, m_d

    def forward(self, x, eff_mask, base_init):
        # x:         [B, in_ch, H, W]
        # eff_mask:  [B, 1, H, W]
        # base_init: [B, 3, H, W]

        # Partial encoder
        e1, m1 = self.penc1(x, eff_mask)
        x2, m2 = self._pool_with_mask(e1, m1)
        e2, m2 = self.penc2(x2, m2)
        x3, m3 = self._pool_with_mask(e2, m2)
        e3, m3 = self.penc3(x3, m3)
        x4, m4 = self._pool_with_mask(e3, m3)
        e4, m4 = self.penc4(x4, m4)
        xb, mb = self._pool_with_mask(e4, m4)

        b = self.bottleneck(xb)

        d4 = self.dec4(torch.cat([self.up(b),  e4, m4], dim=1))
        d3 = self.dec3(torch.cat([self.up(d4), e3, m3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2, m2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1, m1], dim=1))

        residual = self.out_residual(d1)
        pred = (base_init + residual).clamp(0.0, 1.0)

        out = {"pred": pred, "residual": residual}
        if self.predict_confidence:
            out["confidence"] = torch.sigmoid(self.out_conf(d1))
        return out


def consensus_unet_sanity():
    m = ConsensusUNet(in_ch=CFG.in_channels, base=CFG.base_channels)
    x = torch.randn(2, CFG.in_channels, 256, 256)
    mask = (torch.rand(2, 1, 256, 256) > 0.2).float()
    base = torch.rand(2, 3, 256, 256)
    out = m(x, mask, base)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"ConsensusUNet параметров: {n_params/1e6:.2f}M")
    print(f"  pred:       {tuple(out['pred'].shape)}")
    print(f"  residual:   {tuple(out['residual'].shape)}")
    if 'confidence' in out:
        print(f"  confidence: {tuple(out['confidence'].shape)}  mean={out['confidence'].mean():.3f}")
    diff = (out['pred'] - base.clamp(0, 1)).abs().max().item()
    print(f"  |pred - base|_max при zero-init: {diff:.6f}  (должно быть ~0)")

consensus_unet_sanity()

## 6. Метрика, лосс, EMA

In [ ]:
@torch.no_grad()
def psnr_uint8(pred, gt):
    # PSNR по uint8 — как при реальной оценке
    pred_u8 = (pred.clamp(0, 1) * 255.0).round()
    gt_u8   = (gt.clamp(0, 1)   * 255.0).round()
    mse = ((pred_u8 - gt_u8) ** 2).mean().item()
    if mse < 1e-10:
        return 99.0
    return 20.0 * math.log10(255.0 / math.sqrt(mse))


@torch.no_grad()
def psnr_float(pred, gt):
    mse = ((pred - gt) ** 2).mean().item()
    if mse < 1e-10:
        return 99.0
    return 20.0 * math.log10(1.0 / math.sqrt(mse))


def consensus_loss(pred, gt, eff_mask, w_valid=1.0, w_hole=1.0):
    # Взвешенный MSE по effective_mask.
    # w_valid — вес для пикселей с данными (mask=1)
    # w_hole  — вес для дырок (mask=0)
    sq = (pred - gt) ** 2
    w = w_valid * eff_mask + w_hole * (1.0 - eff_mask)
    return (sq * w).mean()


class ModelEMA:
    def __init__(self, model, decay=0.9995):
        self.decay = decay
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        d = self.decay
        for ep, p in zip(self.module.parameters(), model.parameters()):
            ep.data.mul_(d).add_(p.data, alpha=1 - d)
        for eb, b in zip(self.module.buffers(), model.buffers()):
            eb.data.copy_(b.data)

## 7. Бейзлайны без модели — PSNR от warp / base_init / mean(t0,t1)

Полезно знать, насколько модель должна вообще что-то добавить, прежде чем
запускать длинное обучение.

In [ ]:
@torch.no_grad()
def measure_baselines(loader, device):
    warp_psnrs, base_psnrs, mean_psnrs = [], [], []
    for batch in loader:
        inputs = batch["inputs"].to(device)
        base = batch["base_init"].to(device)
        gt = batch["target"].to(device)
        warp_rgb = inputs[:, 0:3]
        has_mean = inputs.size(1) >= 13
        for i in range(gt.size(0)):
            warp_psnrs.append(psnr_uint8(warp_rgb[i:i + 1], gt[i:i + 1]))
            base_psnrs.append(psnr_uint8(base[i:i + 1], gt[i:i + 1]))
            if has_mean:
                mean_psnrs.append(psnr_uint8(inputs[i:i + 1, 10:13], gt[i:i + 1]))
    out = {
        "warp_only": float(np.mean(warp_psnrs)),
        "base_init": float(np.mean(base_psnrs)),
    }
    if mean_psnrs:
        out["mean_t0t1"] = float(np.mean(mean_psnrs))
    return out


def print_baselines(bl: dict, title="Baselines (no model)"):
    print(f"\n--- {title} ---")
    order = ["warp_only", "mean_t0t1", "base_init"]
    for k in order:
        if k in bl:
            print(f"  {k}: {bl[k]:.3f} dB")
    for k, v in bl.items():
        if k not in order:
            print(f"  {k}: {v:.3f} dB")
    print()

## 8. Train / Val split

In [ ]:
def make_split(all_samples, val_fraction, seed=42):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(all_samples))
    rng.shuffle(idx)
    n_val = int(len(all_samples) * val_fraction)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    train = [all_samples[i] for i in train_idx]
    val   = [all_samples[i] for i in val_idx]
    return train, val


def make_dataloaders(train_ds, val_ds, cfg):
    """Единая сборка DataLoader — без persistent_workers на Windows."""
    nw = cfg.num_workers
    pin = DEVICE.type == "cuda"
    common = dict(num_workers=nw, pin_memory=pin)
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        drop_last=True, **common,
    )
    val_loader = DataLoader(
        val_ds, batch_size=1, shuffle=False, **common,
    )
    return train_loader, val_loader

## 9. Training loop

In [ ]:
def cosine_with_warmup(step, total_steps, warmup_steps, base_lr, min_lr=1e-6):
    if step < warmup_steps:
        return base_lr * (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, device, use_amp=True, amp_dtype=torch.bfloat16):
    model.eval()
    psnrs = []
    for batch in loader:
        inputs   = batch["inputs"].to(device, non_blocking=True)
        eff_mask = batch["effective_mask"].to(device, non_blocking=True)
        base     = batch["base_init"].to(device, non_blocking=True)
        gt       = batch["target"].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            out = model(inputs, eff_mask, base)
            pred = out["pred"].float()
        for i in range(pred.size(0)):
            psnrs.append(psnr_uint8(pred[i:i+1], gt[i:i+1]))
    return float(np.mean(psnrs))


def train_one_run(model, train_loader, val_loader, cfg, tag="consensus"):
    device = DEVICE
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                                  weight_decay=cfg.weight_decay,
                                  betas=(0.9, 0.999))
    amp_dtype = torch.bfloat16 if cfg.amp_dtype == "bf16" else torch.float16
    scaler = torch.amp.GradScaler(device.type) if (cfg.use_amp and amp_dtype == torch.float16) else None
    ema = ModelEMA(model, decay=cfg.ema_decay) if cfg.use_ema else None

    steps_per_epoch = len(train_loader)
    total_steps = cfg.num_epochs * steps_per_epoch
    warmup_steps = cfg.warmup_epochs * steps_per_epoch

    history = {"train_psnr": [], "val_psnr": [], "val_psnr_ema": [], "lr": []}
    best_psnr = -1.0
    best_ckpt_path = None
    global_step = 0

    for epoch in range(cfg.num_epochs):
        model.train()
        ep_train_psnr = []
        for batch in train_loader:
            lr = cosine_with_warmup(global_step, total_steps, warmup_steps, cfg.lr)
            for g in optimizer.param_groups:
                g["lr"] = lr

            inputs   = batch["inputs"].to(device, non_blocking=True)
            eff_mask = batch["effective_mask"].to(device, non_blocking=True)
            base     = batch["base_init"].to(device, non_blocking=True)
            gt       = batch["target"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=cfg.use_amp):
                out = model(inputs, eff_mask, base)
                loss = consensus_loss(out["pred"], gt, eff_mask,
                                      w_valid=cfg.loss_weight_valid,
                                      w_hole=cfg.loss_weight_hole)

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                optimizer.step()

            if ema is not None:
                ema.update(model)

            with torch.no_grad():
                ep_train_psnr.append(psnr_float(out["pred"].float(), gt))
            global_step += 1

        train_psnr   = float(np.mean(ep_train_psnr))
        val_psnr     = evaluate(model, val_loader, device, cfg.use_amp, amp_dtype)
        val_psnr_ema = evaluate(ema.module, val_loader, device, cfg.use_amp, amp_dtype) if ema else val_psnr

        history["train_psnr"].append(train_psnr)
        history["val_psnr"].append(val_psnr)
        history["val_psnr_ema"].append(val_psnr_ema)
        history["lr"].append(lr)

        marker = ""
        score = max(val_psnr, val_psnr_ema)
        if score > best_psnr:
            best_psnr = score
            ckpt_path = Path(cfg.save_dir) / f"{tag}_best.pt"
            torch.save({
                "model": model.state_dict(),
                "ema": ema.module.state_dict() if ema else None,
                "epoch": epoch,
                "val_psnr": val_psnr,
                "val_psnr_ema": val_psnr_ema,
                "config": cfg.__dict__,
            }, ckpt_path)
            best_ckpt_path = ckpt_path
            marker = " *"

        print(f"[{tag}] epoch {epoch+1:3d}/{cfg.num_epochs}  "
              f"lr={lr:.2e}  train={train_psnr:.3f}  val={val_psnr:.3f}  "
              f"val_ema={val_psnr_ema:.3f}{marker}")

    print(f"\n[{tag}] BEST PSNR: {best_psnr:.3f} dB  ({best_ckpt_path})")
    return {"history": history, "best_psnr": best_psnr, "best_ckpt": best_ckpt_path}

## 10. SANITY CHECK — overfit на горстке сэмплов

Если модель не выходит **>40 dB** на 8 сэмплах за 300 эпох — что-то не так с
архитектурой или данными. Если выходит — capacity достаточно, и можно переходить
к большому датасету.

In [ ]:
def sanity_overfit_test(n_samples=8, n_epochs=300):
    samples = ALL_SAMPLES if len(ALL_SAMPLES) else discover_samples(CFG)
    if len(samples) == 0:
        print("Нет данных! Проверь пути в CFG.")
        return
    subset = samples[:n_samples]
    ds = ConsensusDataset(subset, CFG, patch_size=None, augment=False)
    loader = DataLoader(ds, batch_size=min(n_samples, 4), shuffle=True,
                        num_workers=0, drop_last=False)

    print_baselines(measure_baselines(loader, DEVICE), title="Baselines (sanity subset)")

    cfg = Config(num_epochs=n_epochs, warmup_epochs=10, lr=5e-4,
                 weight_decay=0.0, use_ema=False,
                 save_dir="./sanity_consensus")
    Path(cfg.save_dir).mkdir(parents=True, exist_ok=True)
    model = ConsensusUNet(in_ch=cfg.in_channels, base=cfg.base_channels).to(DEVICE)
    result = train_one_run(model, loader, loader, cfg, tag="sanity")
    print(f"\n  Если best_psnr >= 40 dB — модель в порядке.")
    return result

#sanity_result = sanity_overfit_test(n_samples=8, n_epochs=300)

## 11. Бейзлайн-тренировка на полном датасете

- **§11** — `ConsensusUNet` (`run_baseline_unet`)
- **§14** — `ConsensusMARNet` (`run_baseline_marnet`)

Перед обучением печатаются baselines: `warp_only`, `mean_t0t1`, `base_init`.

In [ ]:
def run_training(model_factory, num_epochs=200, tag="model"):
    """Общий запуск: split → baselines → train."""
    samples = ALL_SAMPLES if len(ALL_SAMPLES) else discover_samples(CFG)
    print(f"Всего сэмплов: {len(samples)}")
    train_paths, val_paths = make_split(samples, CFG.val_fraction, CFG.seed)
    print(f"  train={len(train_paths)}  val={len(val_paths)}")

    train_ds = ConsensusDataset(train_paths, CFG, patch_size=CFG.patch_size, augment=True)
    val_ds = ConsensusDataset(val_paths, CFG, patch_size=None, augment=False)
    train_loader, val_loader = make_dataloaders(train_ds, val_ds, CFG)

    print_baselines(measure_baselines(val_loader, DEVICE))

    cfg = Config(**{**CFG.__dict__, "num_epochs": num_epochs})
    model = model_factory(cfg)
    return train_one_run(model, train_loader, val_loader, cfg, tag=tag)


def run_baseline_unet(num_epochs=200):
    return run_training(
        lambda cfg: ConsensusUNet(in_ch=cfg.in_channels, base=cfg.base_channels),
        num_epochs=num_epochs,
        tag="consensus_unet",
    )


#baseline_result = run_baseline_unet(num_epochs=200)

## 12. TTA inference (8× геометрических аугментаций)

In [ ]:
@torch.no_grad()
def tta_predict(model, inputs, eff_mask, base, use_amp=True, amp_dtype=torch.bfloat16):
    model.eval()
    device = inputs.device
    preds = []

    def fwd(x_, m_, b_):
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            return model(x_, m_, b_)["pred"].float()

    for k in range(4):
        for flip in (False, True):
            x_aug = torch.rot90(inputs,   k=k, dims=(-2, -1))
            m_aug = torch.rot90(eff_mask, k=k, dims=(-2, -1))
            b_aug = torch.rot90(base,     k=k, dims=(-2, -1))
            if flip:
                x_aug = torch.flip(x_aug, dims=(-1,))
                m_aug = torch.flip(m_aug, dims=(-1,))
                b_aug = torch.flip(b_aug, dims=(-1,))

            p = fwd(x_aug, m_aug, b_aug)
            if flip:
                p = torch.flip(p, dims=(-1,))
            p = torch.rot90(p, k=-k, dims=(-2, -1))
            preds.append(p)

    return torch.stack(preds, dim=0).mean(dim=0)


@torch.no_grad()
def evaluate_with_tta(model, loader, device, use_amp=True, amp_dtype=torch.bfloat16):
    model.eval()
    psnrs = []
    for batch in loader:
        inputs   = batch["inputs"].to(device)
        eff_mask = batch["effective_mask"].to(device)
        base     = batch["base_init"].to(device)
        gt       = batch["target"].to(device)
        pred = tta_predict(model, inputs, eff_mask, base, use_amp, amp_dtype)
        for i in range(pred.size(0)):
            psnrs.append(psnr_uint8(pred[i:i+1], gt[i:i+1]))
    return float(np.mean(psnrs))

## 13. Визуализация

In [ ]:
def to_numpy_img(t):
    if t.dim() == 4:
        t = t[0]
    if t.size(0) == 1:
        return t.cpu().numpy()[0]
    return t.detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()


@torch.no_grad()
def visualize_sample(model, batch, device, use_amp=True, amp_dtype=torch.bfloat16):
    model.eval()
    inputs   = batch["inputs"].to(device)
    eff_mask = batch["effective_mask"].to(device)
    base     = batch["base_init"].to(device)
    gt       = batch["target"].to(device)
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        out = model(inputs, eff_mask, base)
    pred = out["pred"].float()
    warp_rgb = inputs[:, 0:3]
    coverage = inputs[:, 3:4]
    depth    = inputs[:, 4:5]
    sf_rgb   = inputs[:, 5:8]
    sf_mask  = inputs[:, 8:9]
    art_mask = inputs[:, 9:10]
    mean_t0t1 = inputs[:, 10:13] if inputs.size(1) >= 13 else None

    err = (pred - gt).abs().mean(dim=1, keepdim=True)

    psnr = psnr_uint8(pred, gt)
    psnr_warp = psnr_uint8(warp_rgb, gt)
    psnr_base = psnr_uint8(base, gt)

    fig, axes = plt.subplots(3, 3, figsize=(15, 9))
    axes[0,0].imshow(to_numpy_img(gt));        axes[0,0].set_title("Target (GT)")
    axes[0,1].imshow(to_numpy_img(warp_rgb));  axes[0,1].set_title(f"Raw warp  PSNR={psnr_warp:.2f}")
    axes[0,2].imshow(to_numpy_img(base));      axes[0,2].set_title(f"Base init  PSNR={psnr_base:.2f}")

    axes[1,0].imshow(to_numpy_img(pred));      axes[1,0].set_title(f"Prediction  PSNR={psnr:.2f}")
    axes[1,1].imshow(to_numpy_img(coverage), cmap="viridis", vmin=0, vmax=1)
    axes[1,1].set_title("Coverage")
    axes[1,2].imshow(to_numpy_img(sf_rgb));    axes[1,2].set_title("Static far")

    axes[2,0].imshow(to_numpy_img(err), cmap="hot", vmin=0, vmax=0.2)
    axes[2,0].set_title("|pred-gt| (channel mean)")
    axes[2,1].imshow(to_numpy_img(depth), cmap="plasma")
    axes[2,1].set_title("Depth (norm)")
    if mean_t0t1 is not None:
        axes[2, 2].imshow(to_numpy_img(mean_t0t1))
        axes[2, 2].set_title("mean(t0,t1)")
    else:
        axes[2, 2].imshow(to_numpy_img(art_mask), cmap="gray", vmin=0, vmax=1)
        axes[2, 2].set_title("Artifact mask")

    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()
    return {"psnr": psnr, "psnr_warp": psnr_warp, "psnr_base": psnr_base}

## 14. Альтернатива: ConsensusMARNet — SPADE-модуляция

Идея — вместо PartialConv поднимаем модель за счёт **модальной фьюзии**:
- два **shared** энкодера: один прогоняет `warp_rgb`, другой `static_far_rgb`;
- на каждом масштабе **SPADE-модуляция** признаков по карте мета-инфы
  (`coverage`, `depth`, `static_far_mask`, `artifact_mask`) — депф/маски
  предсказывают γ и β для нормализации RGB-фич;
- две RGB-ветки сливаются через `Conv2d(2C → C)` на каждом масштабе;
- декодер обычный, residual поверх `base_init` с zero-init.

Эту архитектуру можно сравнить с `ConsensusUNet` на одном валид-сплите.

In [ ]:
class SPADE(nn.Module):
    # γ, β из meta-карты модулируют нормированные RGB-фичи. zero-init → identity.
    def __init__(self, rgb_ch, meta_ch, mid_ch=None):
        super().__init__()
        mid_ch = mid_ch or max(rgb_ch // 2, 16)
        self.norm = nn.GroupNorm(num_groups=min(8, rgb_ch), num_channels=rgb_ch, affine=False)
        self.shared = nn.Sequential(
            nn.Conv2d(meta_ch, mid_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.to_gamma = nn.Conv2d(mid_ch, rgb_ch, 3, padding=1)
        self.to_beta  = nn.Conv2d(mid_ch, rgb_ch, 3, padding=1)
        nn.init.zeros_(self.to_gamma.weight); nn.init.zeros_(self.to_gamma.bias)
        nn.init.zeros_(self.to_beta.weight);  nn.init.zeros_(self.to_beta.bias)

    def forward(self, rgb_feat, meta_feat):
        if meta_feat.shape[-2:] != rgb_feat.shape[-2:]:
            meta_feat = F.interpolate(meta_feat, size=rgb_feat.shape[-2:],
                                      mode="bilinear", align_corners=False)
        h = self.shared(meta_feat)
        gamma = self.to_gamma(h); beta = self.to_beta(h)
        return self.norm(rgb_feat) * (1.0 + gamma) + beta


class FrameEncoder(nn.Module):
    # 4-уровневый encoder, возвращает фичи на 4 масштабах
    def __init__(self, in_ch, base):
        super().__init__()
        self.b0 = ConvBlock(in_ch, base)
        self.b1 = ConvBlock(base, base * 2)
        self.b2 = ConvBlock(base * 2, base * 4)
        self.b3 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        f0 = self.b0(x)
        f1 = self.b1(self.pool(f0))
        f2 = self.b2(self.pool(f1))
        f3 = self.b3(self.pool(f2))
        return [f0, f1, f2, f3]


class ConsensusMARNet(nn.Module):
    def __init__(self, base=32, meta_base=16, in_channels=13):
        super().__init__()
        self.in_channels = in_channels
        self.n_rgb_branches = 3 if in_channels >= 13 else 2
        self.rgb_ch  = [base, base * 2, base * 4, base * 8]
        self.meta_ch = [meta_base, meta_base * 2, meta_base * 4, meta_base * 8]

        self.rgb_enc  = FrameEncoder(in_ch=3, base=base)
        self.meta_enc = FrameEncoder(in_ch=4, base=meta_base)

        self.spades = nn.ModuleList([
            SPADE(self.rgb_ch[s], self.meta_ch[s]) for s in range(4)
        ])
        self.fuse = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(self.n_rgb_branches * self.rgb_ch[s], self.rgb_ch[s], 3, padding=1),
                nn.GELU(),
                nn.Conv2d(self.rgb_ch[s], self.rgb_ch[s], 3, padding=1),
                nn.GELU(),
            ) for s in range(4)
        ])

        self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec3 = ConvBlock(self.rgb_ch[3] + self.rgb_ch[2], self.rgb_ch[2])
        self.dec2 = ConvBlock(self.rgb_ch[2] + self.rgb_ch[1], self.rgb_ch[1])
        self.dec1 = ConvBlock(self.rgb_ch[1] + self.rgb_ch[0], self.rgb_ch[0])

        self.out_residual = nn.Conv2d(self.rgb_ch[0], 3, 3, padding=1)
        nn.init.zeros_(self.out_residual.weight)
        nn.init.zeros_(self.out_residual.bias)

    def _split(self, x):
        warp     = x[:, 0:3]
        coverage = x[:, 3:4]
        depth    = x[:, 4:5]
        sf_rgb   = x[:, 5:8]
        sf_mask  = x[:, 8:9]
        art_mask = x[:, 9:10]
        meta = torch.cat([coverage, depth, sf_mask, art_mask], dim=1)
        anchors = []
        if x.size(1) >= 13:
            anchors = [x[:, 10:13]]
        return warp, sf_rgb, meta, anchors

    def forward(self, x, eff_mask, base_init):
        warp_rgb, sf_rgb, meta, anchors = self._split(x)
        rgb_inputs = [warp_rgb, sf_rgb] + anchors

        all_feats = [self.rgb_enc(ri) for ri in rgb_inputs]
        meta_feats = self.meta_enc(meta)

        merged = []
        for s in range(4):
            mods = [self.spades[s](all_feats[i][s], meta_feats[s]) for i in range(len(rgb_inputs))]
            merged.append(self.fuse[s](torch.cat(mods, dim=1)))

        d3 = merged[3]
        d2 = self.dec3(torch.cat([self.up(d3), merged[2]], dim=1))
        d1 = self.dec2(torch.cat([self.up(d2), merged[1]], dim=1))
        d0 = self.dec1(torch.cat([self.up(d1), merged[0]], dim=1))

        residual = self.out_residual(d0)
        pred = (base_init + residual).clamp(0, 1)
        return {"pred": pred, "residual": residual}


def consensus_marnet_sanity():
    m = ConsensusMARNet(base=32, meta_base=16, in_channels=CFG.in_channels)
    x = torch.randn(2, CFG.in_channels, 256, 256)
    mask = (torch.rand(2, 1, 256, 256) > 0.2).float()
    base = torch.rand(2, 3, 256, 256)
    out = m(x, mask, base)
    n = sum(p.numel() for p in m.parameters())
    print(f"ConsensusMARNet параметров: {n/1e6:.2f}M")
    print(f"  pred: {tuple(out['pred'].shape)}")
    diff = (out["pred"] - base.clamp(0,1)).abs().max().item()
    print(f"  |pred - base|_max при zero-init: {diff:.6f}  (должно быть ~0)")

consensus_marnet_sanity()


def run_baseline_marnet(num_epochs=200):
    return run_training(
        lambda cfg: ConsensusMARNet(base=cfg.base_channels, in_channels=cfg.in_channels),
        num_epochs=num_epochs,
        tag="consensus_marnet",
    )


# Запуск MARNet (нужны ячейки run_training + класс выше)
marnet_result = run_baseline_marnet(num_epochs=200)

### Сравнение архитектур

```python
def run_arch_comparison(num_epochs=100):
    samples = ALL_SAMPLES if len(ALL_SAMPLES) else discover_samples(CFG)
    train_paths, val_paths = make_split(samples, CFG.val_fraction, CFG.seed)
    train_ds = ConsensusDataset(train_paths, CFG, patch_size=CFG.patch_size, augment=True)
    val_ds   = ConsensusDataset(val_paths,   CFG, patch_size=None,           augment=False)
    train_loader, val_loader = make_dataloaders(train_ds, val_ds, CFG)
    print_baselines(measure_baselines(val_loader, DEVICE))

    cfg = Config(**{**CFG.__dict__, "num_epochs": num_epochs})
    results = {}
    for name, ctor in [
        ("unet_pconv", lambda: ConsensusUNet(in_ch=cfg.in_channels, base=cfg.base_channels)),
        ("marnet",     lambda: ConsensusMARNet(base=cfg.base_channels, in_channels=cfg.in_channels)),
    ]:
        print(f"\n=== {name} ===")
        results[name] = train_one_run(ctor(), train_loader, val_loader, cfg, tag=name)

    print("\n=== ИТОГ ===")
    for name, r in results.items():
        print(f"  {name}: best PSNR = {r['best_psnr']:.3f} dB")
    return results

# results = run_arch_comparison(num_epochs=100)
```

## 15. Ablations / следующие шаги

**Что попробовать в первую очередь:**

1. **Residual gate** — `pred = base + residual * (1 - coverage)`: модель вообще
   не трогает уверенные пиксели. Часто +0.2-0.5 dB на восстановлении дырок без
   просадки на покрытых.
2. **Loss weighting** — `loss_weight_hole = 2.0 ... 3.0` чтобы модель сильнее
   училась восполнять дырки. Следи, чтобы не страдал PSNR на видимых пикселях.
3. **Анизотропный блюр в inference** в направлении ego-motion
   (см. precompute с `compute_per_pixel_motion`) — даёт +0.1-0.3 dB за счёт
   смещения распределения ошибки к минимуму MSE.
4. **Patch curriculum** — старт с patch=128, потом 256, потом 384.

**Что НЕ нужно:**
- Никаких цветовых аугментаций (портит target-RGB, вредит PSNR).
- Не подавать RIFE на вход — мы специально от него ушли, чтобы не учить ошибки.
- L1 как основной лосс — для PSNR-метрики MSE строго лучше.

**Полезные проверки:**
- `print_baselines(measure_baselines(val_loader, DEVICE))` — `warp_only`, `mean_t0t1`, `base_init`.
  Если `base_init >> warp_only`, static_far / mean дают буст уже на старте.
- Sanity overfit ≥ 40 dB на 8 сэмплах — capacity OK.
- Если `train_psnr ≈ val_psnr` на полном датасете — недообучение, увеличивай
  `base_channels` или тренируй дольше.